In [1]:
import torch
torch.cuda.empty_cache()

In [2]:
torch.cuda.ipc_collect()

In [3]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

In [4]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [5]:
!kill -9 626

/bin/bash: line 1: kill: (626) - No such process


In [6]:
torch.cuda.empty_cache()

In [11]:
import torch
import torch.nn as nn
import torchvision.models as models
from timm import create_model

class ResViT(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()

        # ✅ ResNet18 (lightweight)
        self.cnn = models.resnet18(weights="IMAGENET1K_V1")
        cnn_dim = self.cnn.fc.in_features
        self.cnn.fc = nn.Identity()

        # ✅ ViT Small (fixed indentation)
        self.vit = create_model('vit_small_patch16_224', pretrained=True, num_classes=0)
        vit_dim = 384

        # ✅ Classifier
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(cnn_dim + vit_dim),
            nn.Linear(cnn_dim + vit_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        f_cnn = self.cnn(x)
        f_vit = self.vit(x)

        # CONCAT
        f = torch.cat([f_cnn, f_vit], dim=1)

        return self.classifier(f)

In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = ResViT().to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 175MB/s]


model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

In [14]:
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler
import torch
import gc

scaler = GradScaler()

def train_model(model, train_loader, val_loader, epochs=20):
    best_acc = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for images, labels in tqdm(train_loader):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad()

            # ✅ MIXED PRECISION START
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
            # ✅ MIXED PRECISION END

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()

            # ✅ OPTIONAL: free small tensors
            del outputs, loss

        scheduler.step()

        # ✅ Evaluation
        val_acc = evaluate(model, val_loader)

        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}, Val Acc: {val_acc:.4f}")

        # ✅ MEMORY CLEANUP AFTER EACH EPOCH
        gc.collect()
        torch.cuda.empty_cache()

    return model

/tmp/ipykernel_55/2102352208.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [15]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds))

    print("\nConfusion Matrix:")
    print(confusion_matrix(all_labels, all_preds))

    return correct / total

In [16]:
!pip install grad-cam

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 60.1 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for grad-cam: filename=grad_cam-1.5.5-py3-none-any.whl size=44286 sha256=1e277cc40ab6ae65c1a7b92947f54fe70aadf67fe67634a0379f48bcc3be4953
  Stored in directory: /root/.cache/pip/wheels/fb/3b/09/2afc520f3d69bc26ae6bd87416759c820a3f7d05c1a077bbf6
Successfully built grad-cam


In [18]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Use last conv layer of ResNet
target_layers = [model.cnn.layer4[-1]]

cam = GradCAM(model=model, target_layers=target_layers)

def apply_gradcam(image_tensor):
    grayscale_cam = cam(input_tensor=image_tensor.unsqueeze(0))

    cam_image = show_cam_on_image(
        image_tensor.permute(1,2,0).cpu().numpy(),
        grayscale_cam[0],
        use_rgb=True
    )

    return cam_image

In [19]:
data_dir = "/kaggle/input/datasets/venkatsaikondra/multi-venkat/Final_Data"

In [20]:
from torchvision import datasets, transforms

train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

train_dataset = datasets.ImageFolder(
    data_dir + "/train", transform=train_transform
)

val_dataset = datasets.ImageFolder(
    data_dir + "/val", transform=val_transform
)

test_dataset = datasets.ImageFolder(
    data_dir + "/test", transform=val_transform
)

In [21]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)

In [22]:
model = train_model(model, train_loader, val_loader, epochs=20)

test_acc = evaluate(model, test_loader)

print(f"Final Test Accuracy: {test_acc:.4f}")

  0%|          | 0/472 [00:00<?, ?it/s]/tmp/ipykernel_55/2102352208.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
100%|██████████| 472/472 [01:46<00:00,  4.45it/s]



Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.99       404
           1       0.88      0.98      0.93       404
           2       0.69      0.90      0.78       404
           3       0.89      0.52      0.66       404

    accuracy                           0.85      1616
   macro avg       0.86      0.85      0.84      1616
weighted avg       0.86      0.85      0.84      1616


Confusion Matrix:
[[401   3   0   0]
 [  0 396   7   1]
 [  1  15 363  25]
 [  6  34 154 210]]
Epoch 1, Loss: 363.4510, Val Acc: 0.8478


  0%|          | 0/472 [00:00<?, ?it/s]/tmp/ipykernel_55/2102352208.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
100%|██████████| 472/472 [01:09<00:00,  6.81it/s]



Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       404
           1       0.97      0.93      0.95       404
           2       0.80      0.83      0.81       404
           3       0.79      0.79      0.79       404

    accuracy                           0.89      1616
   macro avg       0.89      0.89      0.89      1616
weighted avg       0.89      0.89      0.89      1616


Confusion Matrix:
[[402   0   0   2]
 [  4 377   7  16]
 [  0   3 336  65]
 [  1   7  78 318]]
Epoch 2, Loss: 296.2818, Val Acc: 0.8868


  0%|          | 0/472 [00:00<?, ?it/s]/tmp/ipykernel_55/2102352208.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
100%|██████████| 472/472 [01:09<00:00,  6.78it/s]


...


Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       404
           1       0.98      0.98      0.98       404
           2       0.81      0.91      0.86       404
           3       0.90      0.79      0.84       404

    accuracy                           0.92      1616
   macro avg       0.92      0.92      0.92      1616
weighted avg       0.92      0.92      0.92      1616


Confusion Matrix:
[[402   1   0   1]
 [  0 396   5   3]
 [  0   3 368  33]
 [  0   5  79 320]]
Epoch 18, Loss: 197.2232, Val Acc: 0.9196


  0%|          | 0/472 [00:00<?, ?it/s]/tmp/ipykernel_55/2102352208.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
100%|██████████| 472/472 [01:10<00:00,  6.70it/s]



Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       404
           1       0.98      0.98      0.98       404
           2       0.84      0.86      0.85       404
           3       0.85      0.83      0.84       404

    accuracy                           0.92      1616
   macro avg       0.92      0.92      0.92      1616
weighted avg       0.92      0.92      0.92      1616


Confusion Matrix:
[[403   0   0   1]
 [  0 397   3   4]
 [  0   3 348  53]
 [  0   4  63 337]]
Epoch 19, Loss: 196.1775, Val Acc: 0.9189


  0%|          | 0/472 [00:00<?, ?it/s]/tmp/ipykernel_55/2102352208.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
100%|██████████| 472/472 [01:13<00:00,  6.41it/s]



Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      1.00       404
           1       0.98      0.98      0.98       404
           2       0.84      0.87      0.85       404
           3       0.86      0.83      0.85       404

    accuracy                           0.92      1616
   macro avg       0.92      0.92      0.92      1616
weighted avg       0.92      0.92      0.92      1616


Confusion Matrix:
[[401   1   0   2]
 [  0 395   5   4]
 [  0   3 353  48]
 [  0   3  64 337]]
Epoch 20, Loss: 195.2859, Val Acc: 0.9196

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99       405
           1       0.98      0.98      0.98       405
           2       0.83      0.84      0.83       405
           3       0.83      0.83      0.83       405

    accuracy                           0.91      1620
   macro avg       0.91      0.91      0.91      1620